# Oscillatory game at \(\omega=4\pi\): basic DTB run and sample-size sweep

The deterministic game is

\[
b(x)=\begin{pmatrix}
-\kappa x_1+A\sin(4\pi x_2)\\
-\kappa x_2-A\sin(4\pi x_1)
\end{pmatrix},\qquad A=\kappa=1.
\]

The first section runs an adaptive residual-MMNN DTB trajectory. At every time
step it draws a new random tangent-coordinate subset \(S_k\), solves

\[
\alpha_k=\arg\min_\alpha
\|J_{S_k}(\theta_k,z)\alpha-b(X_k)\|_2^2,
\]

and advances

\[
X_{k+1}=X_k+hJ_{S_k}(\theta_k,z)\alpha_k,
\qquad
\theta_{k+1}[S_k]=\theta_k[S_k]+h\alpha_k.
\]

The second section freezes the **final learned basis** \((\theta_T,S_T)\) and
sweeps the Monte Carlo training-sample size. It uses an independent validation
cloud to measure the finite-sample quantities \(E_N\), \(E_{\rm sample}\),
\(E_G\), and \(E_c\).

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

# Locate DTB_Ver3 locally/on PACE. Clone the branch only when running in Colab.
candidates = [Path.cwd(), *Path.cwd().parents]
repo_root = next((p for p in candidates if (p / 'DTB_Ver3').is_dir()), None)
if repo_root is None:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Run inside the repository or upload the DTB_Ver3 folder.')
    repo_root = Path('/content/dtb-colab-experiments')
    if not repo_root.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', 'codex/game-dynamics-dtb',
            'https://github.com/sun-mengwei/dtb-colab-experiments.git', str(repo_root),
        ], check=True)
sys.path.insert(0, str(repo_root.resolve()))

from DTB_Ver3 import (
    ExperimentConfig,
    OscillatoryNonpotentialGame,
    ResidualMMNN,
    evaluate_dtb_projection,
    run_experiment,
)
from DTB_Ver3.dtb import (
    evaluate_model,
    flat_parameters,
    project_velocity,
    subset_tangent_selection,
)
from DTB_Ver3.models import count_parameters
from DTB_Ver3.utils import (
    relative_l2_error,
    resolve_device,
    resolve_dtype,
    sample_uniform_box,
    warmup_cuda,
    write_csv,
)

output_dir = repo_root / 'DTB_Ver3' / 'results' / 'oscillatory_4pi_basic_and_sample_sweep'
output_dir.mkdir(parents=True, exist_ok=True)
print('repository:', repo_root)
print('output:', output_dir)


## 1. Controls

`TANGENT_SUBSET_SIZE` controls the number of randomly selected MMNN parameter
directions per DTB step. The sample-size section uses nested training clouds
within each seed and an independent validation cloud. Adjust
`SAMPLE_SIZES`, `SAMPLE_SEEDS`, and `VALIDATION_SIZE` before running if needed.

In [ ]:
SEED = 2026
MODEL_SEED = SEED + 100
KAPPA = 1.0
AMPLITUDE = 1.0
OMEGA = 4.0 * np.pi

# Basic DTB trajectory.
N_PARTICLES = 2000
STEP_SIZE = 0.002
FINAL_TIME = 0.2
SNAPSHOT_TIMES = (0.0, 0.05, 0.1, 0.2)
RK4_REFERENCE_STEP = 0.00025

# Residual MMNN and tangent projection.
MMNN_WIDTH = 12
MMNN_RANK = 12
MMNN_DEPTH = 3
ACTIVATION = 'tanh'
TANGENT_SUBSET_SIZE = 128
SVD_RTOL = 1e-8
JACOBIAN_CHUNK = 512

# Fixed-final-basis Monte Carlo sample-size sweep.
SAMPLE_SIZES = (500, 1000, 2000, 5000, 10000, 20000)
SAMPLE_SEEDS = (SEED + 201, SEED + 202, SEED + 203)
VALIDATION_SIZE = 40000
VALIDATION_SEED = SEED + 200

DEVICE_NAME = 'auto'
DTYPE_NAME = 'float64'
DEVICE = resolve_device(DEVICE_NAME)
DTYPE = resolve_dtype(DTYPE_NAME)
warmup_cuda(DEVICE, DTYPE)

print({
    'device': str(DEVICE),
    'dtype': str(DTYPE),
    'omega': OMEGA,
    'particles': N_PARTICLES,
    'dtb_steps': int(round(FINAL_TIME / STEP_SIZE)),
    'tangent_subset_size': TANGENT_SUBSET_SIZE,
    'sample_sizes': SAMPLE_SIZES,
    'sample_seeds': SAMPLE_SEEDS,
    'validation_size': VALIDATION_SIZE,
})


## 2. Basic adaptive DTB run at \(\omega=4\pi\)

A fresh identity-initialized residual MMNN is used. The parameter-coordinate
subset is redrawn at every time step. The tangent inputs are fixed initial
labels \(z\), while the accumulated particles \(X_k\) and parameters
\(\theta_k\) evolve. A refined RK4 trajectory starts from the same particles.

In [ ]:
torch.manual_seed(MODEL_SEED)
model = ResidualMMNN(
    2,
    width=MMNN_WIDTH,
    rank=MMNN_RANK,
    depth=MMNN_DEPTH,
    activation=ACTIVATION,
    dtype=DTYPE,
    zero_init_output=True,
).to(DEVICE)
parameter_count = count_parameters(model)
subset_size = min(TANGENT_SUBSET_SIZE, parameter_count)
game = OscillatoryNonpotentialGame(KAPPA, AMPLITUDE, float(OMEGA))

config = ExperimentConfig(
    dynamics='deterministic',
    run_reference=True,
    reference_integrator='rk4',
    reference_step_size=RK4_REFERENCE_STEP,
    particle_count=N_PARTICLES,
    initial_law='uniform',
    initial_low=-1.0,
    initial_high=1.0,
    step_size=STEP_SIZE,
    final_time=FINAL_TIME,
    snapshot_times=SNAPSHOT_TIMES,
    width=MMNN_WIDTH,
    rank=MMNN_RANK,
    depth=MMNN_DEPTH,
    activation=ACTIVATION,
    model_kind='residual_mmnn',
    zero_init_output=True,
    basis_size=subset_size,
    subset_tangent_selection='resample_each_step',
    tangent_input_mode='fixed_initial_labels',
    track_network_map=False,
    svd_rtol=SVD_RTOL,
    jacobian_chunk=JACOBIAN_CHUNK,
    seed=SEED,
    dtype=DTYPE_NAME,
    device=DEVICE_NAME,
    progress_reports=5,
    output_dir=output_dir / 'basic_dtb_run',
    save_outputs=True,
)

result = run_experiment(game, config, model=model)
print({
    'trainable_coordinates': parameter_count,
    'subset_size': subset_size,
    'steps': len(result.projection_times),
    'initial_relative_projection_error': float(result.relative_projection_error[0]),
    'final_relative_projection_error_at_T': result.final_relative_projection_error,
    'final_alpha_norm_at_T': result.final_alpha_norm,
    'final_DTB_RK4_RMS': result.final_paired_rms,
})


## 3. DTB and RK4 point-cloud snapshots at \(\omega=4\pi\)

The top row shows the accumulated DTB particles and the bottom row shows the
refined RK4 reference from the same initial particles. Every panel uses the
same particle indices, axis limits, and time. This makes deformation of the
cloud and accumulated DTB trajectory error directly visible.

In [ ]:
snapshot_times = sorted(result.dtb_snapshots)
plot_count = min(N_PARTICLES, 3000)
plot_indices = np.arange(plot_count)

# Use common limits across every DTB and RK4 panel.
all_snapshot_clouds = []
for snapshot_time in snapshot_times:
    all_snapshot_clouds.append(result.dtb_snapshots[snapshot_time])
    all_snapshot_clouds.append(result.reference_snapshots[snapshot_time])
all_snapshot_points = np.concatenate(all_snapshot_clouds, axis=0)
x_min, y_min = all_snapshot_points.min(axis=0)
x_max, y_max = all_snapshot_points.max(axis=0)
x_padding = max(0.05 * (x_max - x_min), 0.02)
y_padding = max(0.05 * (y_max - y_min), 0.02)

fig, axes = plt.subplots(
    2,
    len(snapshot_times),
    figsize=(4.0 * len(snapshot_times), 7.2),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
for column, snapshot_time in enumerate(snapshot_times):
    dtb_cloud = result.dtb_snapshots[snapshot_time][plot_indices]
    reference_cloud = result.reference_snapshots[snapshot_time][plot_indices]

    axes[0, column].scatter(
        dtb_cloud[:, 0], dtb_cloud[:, 1],
        s=5, alpha=0.45, color='tab:blue', edgecolors='none',
    )
    axes[1, column].scatter(
        reference_cloud[:, 0], reference_cloud[:, 1],
        s=5, alpha=0.45, color='tab:orange', edgecolors='none',
    )
    axes[0, column].set_title(fr'$t={snapshot_time:g}$')
    axes[1, column].set_xlabel(r'$x_1$')
    for row in range(2):
        axes[row, column].set_xlim(x_min - x_padding, x_max + x_padding)
        axes[row, column].set_ylim(y_min - y_padding, y_max + y_padding)
        axes[row, column].set_aspect('equal', adjustable='box')
        axes[row, column].grid(True, alpha=0.2)

axes[0, 0].set_ylabel(r'DTB $x_2$')
axes[1, 0].set_ylabel(r'RK4 reference $x_2$')
fig.suptitle(r'Point-cloud evolution for $\omega=4\pi$', fontsize=15)
snapshot_figure_path = output_dir / 'dtb_rk4_point_cloud_snapshots.png'
fig.savefig(snapshot_figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', snapshot_figure_path)


## 4. Time-dependent DTB diagnostics

The arrays returned by the time integrator contain projections at
\(t_0,\ldots,t_{K-1}\). The notebook appends the fresh diagnostic computed at
\((X_T,\theta_T,S_T)\), so every projection curve includes its true endpoint
at \(T\).

In [ ]:
diagnostic_times = np.append(result.projection_times, result.times[-1])
rms_projection_history = np.append(
    result.projection_error,
    result.final_projection_error,
)
relative_projection_history = np.append(
    result.relative_projection_error,
    result.final_relative_projection_error,
)
alpha_norm_history = np.append(result.alpha_norm, result.final_alpha_norm)

basic_history_path = write_csv(
    output_dir / 'basic_dtb_diagnostics.csv',
    (
        'time', 'rms_projection_error', 'relative_projection_error',
        'alpha_norm',
    ),
    zip(
        diagnostic_times,
        rms_projection_history,
        relative_projection_history,
        alpha_norm_history,
    ),
)

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axes[0, 0].semilogy(diagnostic_times, alpha_norm_history, color='tab:purple')
axes[0, 0].set(title=r'Coefficient norm $\|\alpha_k\|_2$', xlabel='time', ylabel='norm')
axes[0, 1].semilogy(diagnostic_times, rms_projection_history, color='tab:blue')
axes[0, 1].set(title='RMS projection error', xlabel='time', ylabel='RMS error')
axes[1, 0].semilogy(diagnostic_times, relative_projection_history, color='tab:orange')
axes[1, 0].set(title='Relative projection error', xlabel='time', ylabel='relative error')
axes[1, 1].semilogy(result.times, result.trajectory_rms_error, color='tab:red')
axes[1, 1].set(title='DTB versus refined RK4', xlabel='time', ylabel='paired RMS')
for axis in axes.flat:
    axis.grid(True, alpha=0.3)
basic_figure_path = output_dir / 'basic_dtb_diagnostics.png'
fig.savefig(basic_figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', basic_history_path)
print('saved:', basic_figure_path)


## 5. Freeze the learned final basis for the sample-size study

The sweep does not return to an arbitrary initial tangent space. It freezes the
basis obtained after the basic DTB run:

\[
\theta=\theta_T,\qquad S=S_T.
\]

For independent labels \(z\), it evaluates the final neural map
\(Y=T_{\theta_T}(z)\), the tangent matrix
\(J_S(\theta_T,z)\), and the difficult oscillatory target \(q_{4\pi}(Y)\).
The validation coefficients are

\[
\alpha_{\rm ref}=\arg\min_\alpha
\|J_{\rm ref}\alpha-q_{\rm ref}\|_2^2.
\]

This validation cloud is independent of every training cloud.

In [ ]:
_, parameter_structure = flat_parameters(model)
theta_final = torch.as_tensor(result.final_parameters, dtype=DTYPE, device=DEVICE)
final_selected = torch.as_tensor(
    result.selected_indices,
    dtype=torch.long,
    device=DEVICE,
)

print('Building independent validation tangent matrix...', flush=True)
validation_labels = sample_uniform_box(
    VALIDATION_SIZE,
    2,
    low=-1.0,
    high=1.0,
    dtype=DTYPE,
    device=DEVICE,
    seed=VALIDATION_SEED,
)
validation_states = evaluate_model(
    theta_final,
    validation_labels,
    model,
    parameter_structure,
).detach()
validation_target = game.oscillatory_velocity(validation_states)
_, validation_matrix = subset_tangent_selection(
    theta_final,
    final_selected,
    validation_labels,
    model,
    parameter_structure,
    chunk_size=JACOBIAN_CHUNK,
)
reference_projection = project_velocity(
    validation_matrix,
    validation_target,
    rtol=SVD_RTOL,
)
alpha_reference = reference_projection.alpha
G_reference = validation_matrix.T @ validation_matrix / VALIDATION_SIZE
c_reference = (
    validation_matrix.T @ validation_target.reshape(-1) / VALIDATION_SIZE
)
E_repr = float(reference_projection.relative_residual.item())
print({
    'validation_size': VALIDATION_SIZE,
    'fixed_final_subset_size': int(final_selected.numel()),
    'best_validation_representation_error': E_repr,
    'validation_retained_rank': reference_projection.retained_rank,
})


## 6. Sweep the training-sample size

For each training size and seed, the notebook computes

\[
E_N=\frac{\|J_{\rm ref}\widehat\alpha_N-q_{\rm ref}\|_2}
{\|q_{\rm ref}\|_2},
\qquad
E_{\rm sample}=\frac{\|J_{\rm ref}(\widehat\alpha_N-\alpha_{\rm ref})\|_2}
{\|q_{\rm ref}\|_2},
\]

and

\[
E_G=\frac{\|G_N-G_{\rm ref}\|_2}{\|G_{\rm ref}\|_2},
\qquad
E_c=\frac{\|c_N-c_{\rm ref}\|_2}{\|c_{\rm ref}\|_2}.
\]

Within each seed, one maximum-size cloud is generated and smaller sample sizes
use nested prefixes. The MMNN parameters and final tangent subset stay fixed.

In [ ]:
# -----------------------------------------------------------------------------
# FOUR SAMPLE-SIZE ERRORS COMPUTED BELOW
#
# The final learned MMNN parameters theta_T and the final tangent subset S_T
# are fixed throughout this study. Only the Monte Carlo training cloud changes.
#
# HOW q IS EXTRACTED FROM THE ORIGINAL GAME VELOCITY
#
# The full omega=4*pi game velocity is split into a low-frequency damping part
# and a high-frequency oscillatory part:
#
#   b(y) = d(y) + q(y),
#   d(y) = (-kappa*y_1, -kappa*y_2),
#   q(y) = (A*sin(4*pi*y_2), -A*sin(4*pi*y_1)) = b(y) - d(y).
#
# The DTB trajectory above projects the full target b(X_k). This sample-size
# study deliberately uses only q(T_{theta_T}(z)) so that an easily represented
# damping term cannot hide failure to resolve the oscillations. Thus E_N below
# is a q-only relative validation error, not the full-velocity DTB error.
#
# For a fixed tangent matrix and fixed truncated-SVD subspace, the projection
# operator P_J is linear, so
#
#   (I - P_J)b = (I - P_J)d + (I - P_J)q.
#
# However, the squared full residual is
#
#   ||(I-P_J)b||_2^2
#     = ||(I-P_J)d||_2^2 + ||(I-P_J)q||_2^2
#       + 2*<(I-P_J)d, (I-P_J)q>.
#
# Therefore damping and oscillatory error energies do not generally add unless
# their residuals are orthogonal. The q-only diagnostic isolates one component;
# it is not claimed to equal or be an orthogonal part of the full error.
#
# Notation:
#   J_N       = tangent matrix on N training labels, shape (2N, m)
#   q_N       = oscillatory target q_{4pi}(T_{theta_T}(z)) on those labels
#   alpha_N   = argmin_alpha ||J_N alpha - q_N||_2
#   J_ref     = tangent matrix on the independent validation cloud
#   q_ref     = oscillatory target on the independent validation cloud
#   alpha_ref = argmin_alpha ||J_ref alpha - q_ref||_2
#
# 1. E_N: total out-of-sample projection error of the coefficients learned
#    from N training samples:
#
#       E_N = ||J_ref alpha_N - q_ref||_2 / ||q_ref||_2.
#
#    This includes both the best-achievable representation error of the fixed
#    tangent space and the additional error caused by finite-sample fitting.
#
# 2. E_sample: excess validation error caused only by estimating alpha from N
#    samples rather than from the large validation cloud:
#
#       E_sample = ||J_ref (alpha_N - alpha_ref)||_2 / ||q_ref||_2.
#
#    For an exact orthogonal least-squares projection,
#    E_N^2 = E_repr^2 + E_sample^2, where
#    E_repr = ||J_ref alpha_ref - q_ref||_2 / ||q_ref||_2.
#
# 3. E_G: relative error in the empirical tangent Gram matrix:
#
#       G_N   = J_N^T J_N / N,
#       G_ref = J_ref^T J_ref / N_ref,
#       E_G   = ||G_N - G_ref||_F / ||G_ref||_F.
#
#    G describes tangent-direction magnitudes, correlations, and conditioning.
#
# 4. E_c: relative error in the tangent-target correlation vector:
#
#       c_N   = J_N^T q_N / N,
#       c_ref = J_ref^T q_ref / N_ref,
#       E_c   = ||c_N - c_ref||_2 / ||c_ref||_2.
#
#    c measures how strongly each selected tangent direction aligns with the
#    oscillatory target. Decreasing E_sample, E_G, and E_c as N grows is
#    evidence that finite-sample under-resolution is being reduced.
# -----------------------------------------------------------------------------

sample_rows = []
maximum_sample_size = max(SAMPLE_SIZES)
validation_target_norm = torch.linalg.vector_norm(validation_target)

for sample_seed in SAMPLE_SEEDS:
    print(f'Building nested training tangent: seed={sample_seed}', flush=True)
    training_labels_all = sample_uniform_box(
        maximum_sample_size,
        2,
        low=-1.0,
        high=1.0,
        dtype=DTYPE,
        device=DEVICE,
        seed=sample_seed,
    )
    training_states_all = evaluate_model(
        theta_final,
        training_labels_all,
        model,
        parameter_structure,
    ).detach()
    training_target_all = game.oscillatory_velocity(training_states_all)
    _, training_matrix_all = subset_tangent_selection(
        theta_final,
        final_selected,
        training_labels_all,
        model,
        parameter_structure,
        chunk_size=JACOBIAN_CHUNK,
    )

    for sample_size in SAMPLE_SIZES:
        matrix = training_matrix_all[: 2 * sample_size]
        target = training_target_all[:sample_size]
        fitted = project_velocity(matrix, target, rtol=SVD_RTOL)

        validation_prediction = (
            validation_matrix @ fitted.alpha
        ).reshape_as(validation_target)
        E_N = relative_l2_error(validation_prediction, validation_target)
        sampling_prediction = (
            validation_matrix @ (fitted.alpha - alpha_reference)
        ).reshape_as(validation_target)
        E_sample = float(
            torch.linalg.vector_norm(sampling_prediction)
            .div(validation_target_norm)
            .item()
        )

        G_sample = matrix.T @ matrix / sample_size
        c_sample = matrix.T @ target.reshape(-1) / sample_size
        E_G = relative_l2_error(G_sample, G_reference)
        E_c = relative_l2_error(c_sample, c_reference)
        sample_rows.append([
            sample_size,
            sample_seed,
            E_N,
            E_sample,
            E_G,
            E_c,
            float(torch.linalg.vector_norm(fitted.alpha).item()),
            fitted.retained_rank,
            fitted.condition_number,
        ])
        print(
            f'N={sample_size:6d} seed={sample_seed} '
            f'E_N={E_N:.4e} E_sample={E_sample:.4e}',
            flush=True,
        )

sample_columns = (
    'sample_size', 'seed', 'E_N', 'E_sample', 'E_G', 'E_c',
    'alpha_norm', 'retained_rank', 'condition_number',
)
raw_sweep_path = write_csv(
    output_dir / 'sample_size_sweep_raw.csv',
    sample_columns,
    sample_rows,
)
print('saved:', raw_sweep_path)


## 7. Aggregate the sample-size sweep

The following table and figure report the mean and standard deviation over
`SAMPLE_SEEDS`. The horizontal line in the first panel is the best projection
error measured directly on the independent validation cloud.

In [ ]:
sample_summary_rows = []
for sample_size in SAMPLE_SIZES:
    values = np.asarray(
        [row[2:6] for row in sample_rows if row[0] == sample_size],
        dtype=float,
    )
    means = values.mean(axis=0)
    standard_deviations = values.std(axis=0, ddof=1)
    sample_summary_rows.append([
        sample_size,
        means[0], standard_deviations[0],
        means[1], standard_deviations[1],
        means[2], standard_deviations[2],
        means[3], standard_deviations[3],
    ])

sample_summary_columns = (
    'sample_size',
    'E_N_mean', 'E_N_std',
    'E_sample_mean', 'E_sample_std',
    'E_G_mean', 'E_G_std',
    'E_c_mean', 'E_c_std',
)
summary_sweep_path = write_csv(
    output_dir / 'sample_size_sweep_summary.csv',
    sample_summary_columns,
    sample_summary_rows,
)

summary = np.asarray(sample_summary_rows, dtype=float)
N_values = summary[:, 0]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
metric_specs = (
    (1, 2, r'$E_N$', 'tab:blue'),
    (3, 4, r'$E_{sample}$', 'tab:orange'),
    (5, 6, r'$E_G$', 'tab:green'),
    (7, 8, r'$E_c$', 'tab:red'),
)
for axis, (mean_column, std_column, label, color) in zip(axes.flat, metric_specs):
    axis.errorbar(
        N_values,
        summary[:, mean_column],
        yerr=summary[:, std_column],
        marker='o',
        capsize=3,
        color=color,
    )
    axis.set_xscale('log')
    axis.set_yscale('log')
    axis.set(title=f'{label} versus sample size', xlabel='training samples N', ylabel=label)
    axis.grid(True, which='both', alpha=0.3)
axes[0, 0].axhline(E_repr, color='black', linestyle='--', label=r'$E_{repr}$')
axes[0, 0].legend()

sweep_figure_path = output_dir / 'sample_size_sweep_diagnostics.png'
fig.savefig(sweep_figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', summary_sweep_path)
print('saved:', sweep_figure_path)


## 8. Interpretation

- If \(E_N\) is large for small \(N\) and approaches \(E_{\rm repr}\), the
  final tangent basis is expressive enough but the least-squares coefficients
  are initially under-resolved by Monte Carlo sampling.
- If \(E_{\rm sample}\), \(E_G\), and \(E_c\) decrease with \(N\), this is
  direct evidence of finite-sample error.
- If \(E_N\) stays well above \(E_{\rm repr}\), inspect conditioning and repeat
  with more samples or a different SVD tolerance.
- If \(E_{\rm repr}\) itself is large, increasing sample size cannot repair the
  representation limitation of the learned final tangent basis.

In [ ]:
archive_path = Path(shutil.make_archive(
    str(output_dir),
    'zip',
    root_dir=output_dir,
))
print('result archive:', archive_path)
